# Analyse des résultats — ResNet50 Tomato Disease Classification
**Entraînement GLICID/Nautilus — Job 10265134**

Ce notebook analyse les résultats de l'entraînement du modèle ResNet50 fine-tuné sur le dataset `tomato_final` (12 classes, ~25 000 images).

## 0. Imports et configuration

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

import torch
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
from pathlib import Path
from collections import defaultdict

# Chemins
OUTPUT_DIR = Path("sortie_glicid/results_resnet50")
LOGS_DIR   = Path("sortie_glicid/logs")

# Style global
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print("Fichiers disponibles :")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(" -", f.name)

## 1. Courbes d'apprentissage

In [ ]:
with open(OUTPUT_DIR / "history.json") as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)

# Détection du changement de phase (Phase 1 = epochs 1-5, Phase 2 = epoch 6+ )
PHASE2_START = 5.5  # trait entre epoch 5 et 6

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Courbes d'apprentissage — ResNet50 (14 epochs)", fontweight='bold')

# --- Loss ---
ax = axes[0]
ax.plot(epochs, history['train_loss'], 'o-', label='Train Loss', color='steelblue')
ax.plot(epochs, history['val_loss'],   'o-', label='Val Loss',   color='darkorange')
ax.axvline(PHASE2_START + 0.5, color='gray', linestyle='--', alpha=0.7, label='Début fine-tuning')
ax.set_title('Courbe de Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# --- Macro-F1 ---
ax = axes[1]
ax.plot(epochs, history['train_f1'], 'o-', label='Train Macro-F1', color='steelblue')
ax.plot(epochs, history['val_f1'],   'o-', label='Val Macro-F1',   color='darkorange')
ax.axvline(PHASE2_START + 0.5, color='gray', linestyle='--', alpha=0.7, label='Début fine-tuning')

# Annotation meilleur val F1
best_epoch = int(np.argmax(history['val_f1'])) + 1
best_val   = max(history['val_f1'])
ax.annotate(f'Best val\nF1={best_val:.3f}\n(ep. {best_epoch})',
            xy=(best_epoch, best_val),
            xytext=(best_epoch + 1, best_val - 0.03),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=9)

ax.set_title('Courbe Macro-F1')
ax.set_xlabel('Epoch')
ax.set_ylabel('Macro-F1')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'learning_curves_annotated.png', bbox_inches='tight')
print(f"→ Best val Macro-F1 : {best_val:.4f} à l'epoch {best_epoch}")

### Interprétation des courbes

- **Phase 1 — 5 époques** (couche finale seule, lr=0.001) : convergence rapide,
  train et val restent proches — le modèle adapte rapidement la tête de 
  classification à la nouvelle tâche.
- **Phase 2 — 9 époques** (fine-tuning `layer4 + fc`, lr=0.0001) : la train loss
  continue de baisser fortement tandis que la val loss se stabilise autour de 
  0.30–0.36. C'est le signe classique d'un léger surapprentissage en fin 
  d'entraînement. Le checkpoint sauvegardé correspond au meilleur val Macro-F1 
  obtenu à l'epoch 11 (F1=0.889). L'early stopping se déclenche à l'epoch 14 
  après 3 epochs sans amélioration.

## 2. Métriques finales sur le test set

In [ ]:
with open(OUTPUT_DIR / "metrics_test.json") as f:
    metrics = json.load(f)

print("=" * 45)
print("  RÉSULTATS FINAUX — TEST SET")
print("=" * 45)
print(f"  Test Loss       : {metrics['test_loss']:.4f}")
print(f"  Test Accuracy   : {metrics['test_accuracy']:.4f} ")
print(f"  Test Macro-F1   : {metrics['test_macro_f1']:.4f} ")
print(f"  Best Val Macro-F1: {metrics['best_val_macro_f1']:.4f}")
print("=" * 45)

# Comparaison avec baseline
print("\n📊 Comparaison avec la baseline (rapport S1) :")
print(f"  Accuracy  : 69.0% → {metrics['test_accuracy']*100:.1f}%  (+{metrics['test_accuracy']*100 - 69.0:.1f} pts)")
print(f"  Macro-F1  : 74.0% → {metrics['test_macro_f1']*100:.1f}%  (+{metrics['test_macro_f1']*100 - 74.0:.1f} pts)")

## 3. Rapport de classification par classe

In [ ]:
# Lecture et parsing du classification report
lines = (OUTPUT_DIR / "classification_report.txt").read_text().strip().splitlines()

rows = []
for line in lines:
    parts = line.split()
    if len(parts) >= 5 and parts[-1].isdigit():
        support = int(parts[-1])
        f1       = float(parts[-2])
        recall   = float(parts[-3])
        prec     = float(parts[-4])
        name     = " ".join(parts[:-4])
        if name not in ['accuracy', 'macro avg', 'weighted avg']:
            rows.append({'Classe': name, 'Précision': prec, 'Rappel': recall, 'F1-score': f1, 'Support': support})

df = pd.DataFrame(rows).sort_values('F1-score', ascending=False).reset_index(drop=True)

# Affichage stylé
def color_f1(val):
    if val >= 0.90:   return 'background-color: #c6efce; color: #276221'
    elif val >= 0.85: return 'background-color: #ffeb9c; color: #9c5700'
    else:             return 'background-color: #ffc7ce; color: #9c0006'

styled = df.style.applymap(color_f1, subset=['F1-score']) \
                 .format({'Précision': '{:.3f}', 'Rappel': '{:.3f}', 'F1-score': '{:.3f}'}) \
                 .set_caption("Performances par classe — Test set")
display(styled)

In [ ]:
# Visualisation barplot F1 par classe
df_sorted = df.sort_values('F1-score')

colors = ['#d73027' if v < 0.75 else '#fee090' if v < 0.90 else '#1a9850'
          for v in df_sorted['F1-score']]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_sorted['Classe'], df_sorted['F1-score'], color=colors, edgecolor='white')

# Valeurs sur les barres
for bar, val in zip(bars, df_sorted['F1-score']):
    ax.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha='right', fontsize=9, color='white', fontweight='bold')

ax.axvline(0.893, color='navy', linestyle='--', linewidth=1.5, label=f'Macro-F1 moyen = 0.893')
ax.set_xlabel('F1-score')
ax.set_title('F1-score par classe — Test set (ResNet50)')
ax.set_xlim(0.5, 1.02)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'f1_par_classe.png', bbox_inches='tight')
plt.show()

## 4. Analyse de la matrice de confusion

In [ ]:
from PIL import Image
img = Image.open(OUTPUT_DIR / 'confusion_matrix_test.png')
plt.figure(figsize=(12, 10))
plt.imshow(img)
plt.axis('off')
plt.title('Matrice de confusion — Test set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Tableau des principales confusions — extrait automatiquement ──

from sklearn.metrics import confusion_matrix

# Calcul de la matrice de confusion
cm = confusion_matrix(all_labels, all_preds)

# Extraire toutes les confusions hors diagonale
confusions = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i, j] > 0:
            support   = cm[i].sum()
            nb_errors = cm[i, j]
            pct       = round(nb_errors / support * 100, 1)
            confusions.append({
                'Vraie classe'  : class_names[i],
                'Prédite comme' : class_names[j],
                'Nb erreurs'    : nb_errors,
                'Support'       : support,
                '% erreur'      : pct
            })

df_conf = pd.DataFrame(confusions)
df_conf = df_conf.sort_values('% erreur', ascending=False)

# Garder les 5 pires confusions
df_top = df_conf.head(5).reset_index(drop=True)

# Affichage stylé
def color_pct(val):
    if val >= 20:  return 'background-color: #ffc7ce; color: #9c0006'
    elif val >= 10: return 'background-color: #ffeb9c; color: #9c5700'
    else:           return 'background-color: #c6efce; color: #276221'

display(
    df_top.style
          .applymap(color_pct, subset=['% erreur'])
          .format({'% erreur': '{:.1f}%'})
          .set_caption("Top 5 des confusions inter-classes — Test set")
)

## 5. Synthèse et comparaison avec la baseline

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

categories = ['Accuracy', 'Macro-F1']
baseline   = [0.690, 0.740]
nouveau    = [metrics['test_accuracy'], metrics['test_macro_f1']]

x = np.arange(len(categories))
w = 0.3

b1 = ax.bar(x - w/2, baseline, w, label='Baseline (S1 — 10 classes, 1609 img)', color='#aec6cf', edgecolor='gray')
b2 = ax.bar(x + w/2, nouveau,  w, label='ResNet50 (S2 — 12 classes, 24K img)',  color='#2e86ab', edgecolor='gray')

for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=10)

ax.set_ylim(0.5, 1.0)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.set_ylabel('Score')
ax.set_title('Comparaison baseline vs nouveau modèle')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'comparaison_baseline.png', bbox_inches='tight')
plt.show()

## 6. Visualisation des prédictions

In [ ]:
# ── Chemins ─────────────────────────────────────────────────
MODEL_PATH = Path("sortie_glicid/results_resnet50/best_resnet50_tomato.pth")
CSV_PATH   = Path(r"C:\Users\LARIS-1\Desktop\DJMA\tomato_classification\trainingclassif\data\tomato_only\test_split_final.csv")
CLASS_JSON = Path("sortie_glicid/results_resnet50/class_names.json")
OUTPUT_DIR = Path("sortie_glicid/results_resnet50")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")

# ── Chargement CSV ───────────────────────────────────────────
df = pd.read_csv(CSV_PATH)
print(f"Test set : {len(df)} images, {df['class_name'].nunique()} classes")

# ── Noms de classes ──────────────────────────────────────────
with open(CLASS_JSON) as f:
    class_names = json.load(f)
num_classes = len(class_names)
SHORT = {c: c.replace("tomato_", "").replace("_", " ") for c in class_names}

# ── Dataset ──────────────────────────────────────────────────
class TomatoTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        # Pré-scan : marquer les images corrompues
        self.valid = []
        for path in self.df['filepath']:
            try:
                with Image.open(path) as img:
                    img.verify()
                with Image.open(path) as img:
                    img.load()
                self.valid.append(True)
            except Exception:
                self.valid.append(False)
        print(f"Images corrompues ignorées : {sum(not v for v in self.valid)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if not self.valid[idx]:
            # Image corrompue : tenseur noir + label -1 pour pouvoir filtrer
            return self.transform(Image.new("RGB", (224, 224), 0)), -1, row['filepath']
        img   = Image.open(row['filepath']).convert("RGB")
        label = int(row['label'])
        return self.transform(img), label, row['filepath']

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

test_dataset = TomatoTestDataset(df, transform)
test_loader  = DataLoader(test_dataset, batch_size=64,
                          shuffle=False, num_workers=0)

# ── Chargement modèle ────────────────────────────────────────
model = models.resnet50(weights=None)
model.fc = torch.nn.Sequential(
    torch.nn.Dropout(p=0.5),
    torch.nn.Linear(2048, num_classes)
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()
print(f"Modèle chargé ({num_classes} classes)")

# ── Inférence ────────────────────────────────────────────────
all_preds  = []
all_labels = []
all_confs  = []
all_paths  = []

with torch.no_grad():
    for inputs, labels, paths in test_loader:
        inputs  = inputs.to(DEVICE)
        outputs = model(inputs)
        probs   = torch.softmax(outputs, dim=1)
        confs, preds = probs.max(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_confs.extend(confs.cpu().numpy())
        all_paths.extend(paths)

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_confs  = np.array(all_confs)

# Filtrer les corrompues (label == -1)
valid_mask   = (all_labels != -1)
all_preds    = all_preds[valid_mask]
all_labels   = all_labels[valid_mask]
all_confs    = all_confs[valid_mask]
all_paths    = [p for p, v in zip(all_paths, valid_mask) if v]

correct_mask = (all_preds == all_labels)
error_mask   = ~correct_mask

print(f"\nTotal valides : {len(all_labels)}")
print(f"✓ Correctes   : {correct_mask.sum()}  ({correct_mask.mean()*100:.1f}%)")
print(f"✗ Erreurs     : {error_mask.sum()}   ({error_mask.mean()*100:.1f}%)")

# ── Fonction chargement image ────────────────────────────────
def load_img(path):
    try:
        return Image.open(path).convert("RGB")
    except Exception:
        return Image.new("RGB", (224, 224), color=(200, 200, 200))

# ── Grille bonnes prédictions par classe ─────────────────────
N_SAMPLE = 4

fig, axes = plt.subplots(num_classes, N_SAMPLE,
                         figsize=(N_SAMPLE * 3.2, num_classes * 3.2))
fig.suptitle("Bonnes prédictions — exemples par classe",
             fontsize=14, fontweight='bold', y=1.005)

for i, cls_name in enumerate(class_names):
    idx_correct = np.where((all_labels == i) & correct_mask)[0]
    top_idx     = idx_correct[np.argsort(all_confs[idx_correct])[::-1][:N_SAMPLE]]

    for j in range(N_SAMPLE):
        ax = axes[i][j]
        if j < len(top_idx):
            idx = top_idx[j]
            ax.imshow(load_img(all_paths[idx]))
            ax.set_title(
                f"Vrai : {SHORT[class_names[all_labels[idx]]]}\n"
                f"Prédit : {SHORT[class_names[all_preds[idx]]]}\n"
                f"conf = {all_confs[idx]:.2f}",
                fontsize=7, color='green'
            )
        ax.axis('off')

    axes[i][0].set_ylabel(SHORT[cls_name], fontsize=8, rotation=0,
                          labelpad=90, va='center', color='navy')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'bonnes_predictions_par_classe.png',
            bbox_inches='tight', dpi=120)
plt.show()
print("→ Sauvegardé : bonnes_predictions_par_classe.png")

# ── Grille d'erreurs ─────────────────────────────────────────
N_ERRORS = 18

error_idx  = np.where(error_mask)[0]
error_idx  = error_idx[np.argsort(all_confs[error_idx])[::-1]]
sample_idx = error_idx[:N_ERRORS]

n_cols = 6
n_rows = int(np.ceil(N_ERRORS / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 3, n_rows * 3.6))
fig.suptitle("Erreurs de classification (triées par confiance décroissante)",
             fontsize=13, fontweight='bold')

for k, idx in enumerate(sample_idx):
    ax = axes[k // n_cols][k % n_cols]
    ax.imshow(load_img(all_paths[idx]))
    ax.set_title(
        f"Vrai : {SHORT[class_names[all_labels[idx]]]}\n"
        f"Prédit : {SHORT[class_names[all_preds[idx]]]}\n"
        f"conf = {all_confs[idx]:.2f}",
        fontsize=7, color='darkred'
    )
    ax.axis('off')

for k in range(len(sample_idx), n_rows * n_cols):
    axes[k // n_cols][k % n_cols].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'erreurs_classification.png',
            bbox_inches='tight', dpi=120)
plt.show()
print("→ Sauvegardé : erreurs_classification.png")

# ── Erreurs les plus confiantes ──────────────────────────────
N_CONF       = 12
top_conf_idx = error_idx[:N_CONF]

n_cols = 4
n_rows = int(np.ceil(N_CONF / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 3.2, n_rows * 3.8))
fig.suptitle("Erreurs les plus confiantes",
             fontsize=13, fontweight='bold', color='darkred')

for k, idx in enumerate(top_conf_idx):
    ax = axes[k // n_cols][k % n_cols]
    ax.imshow(load_img(all_paths[idx]))

    for spine in ax.spines.values():
        spine.set_edgecolor('red')
        spine.set_linewidth(3)

    ax.set_title(
        f"Vrai : {SHORT[class_names[all_labels[idx]]]}\n"
        f"Prédit : {SHORT[class_names[all_preds[idx]]]}\n"
        f"conf = {all_confs[idx]:.3f}",
        fontsize=8, color='darkred', fontweight='bold'
    )
    ax.axis('off')

for k in range(len(top_conf_idx), n_rows * n_cols):
    axes[k // n_cols][k % n_cols].axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'erreurs_confiantes.png',
            bbox_inches='tight', dpi=120)
plt.show()
print("→ Sauvegardé : erreurs_confiantes.png")

## 7. Figures exportées pour le rapport LaTeX

Les figures suivantes sont disponibles dans `sortie_glicid/results_resnet50/` :

| Fichier | Usage LaTeX |
|---|---|
| `learning_curves_annotated.png` | Courbes loss / F1 avec annotations |
| `confusion_matrix_test.png` | Matrice de confusion |
| `f1_par_classe.png` | Barplot F1 par classe |
| `comparaison_baseline.png` | Comparaison S1 vs S2 |

```latex
\begin{figure}[H]
\centering
\includegraphics[width=0.95\textwidth]{learning_curves_annotated.png}
\caption{Courbes d'apprentissage -- ResNet50, 14 epochs, 12 classes.}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.85\textwidth]{f1_par_classe.png}
\caption{F1-score par classe sur le jeu de test (Macro-F1 = 0.893).}
\end{figure}

\begin{figure}[H]
\centering
\includegraphics[width=0.7\textwidth]{comparaison_baseline.png}
\caption{Comparaison des performances entre la baseline et le nouveau modèle.}
\end{figure}
```

## 7. Conclusions

### Points forts
- **Accuracy 91.5%** et **Macro-F1 89.3%** sur 12 classes — progression significative par rapport à la baseline (69% / 74%)
- 9 classes sur 12 dépassent un F1 de 0.87
- Excellents résultats sur `tomato_bacterial_spot` (F1=0.954), `tomato_late_blight` (0.975), `tomato_mosaic_virus` (0.974), `tomato_yellow_leaf_curl_virus` (0.969)

### Limites identifiées
- `tomato_verticillium_wilt` : F1=0.664 — classe sous-représentée (support=116 en test) avec symptômes diffus
- `tomato_leaf_blight` : F1=0.709 — forte confusion avec `septoria_leaf_spot` (signatures visuelles proches)
- Léger surapprentissage en phase de fine-tuning (écart croissant train/val loss après epoch 6)

### Perspectives pour la suite
- Enrichir `verticillium_wilt` avec plus de données réelles
- Explorer des stratégies de poids de classes plus agressives pour les classes faibles
- Intégrer ce CNN dans le pipeline Phase 1 (SAM3 + LLM + NLI)